In [127]:
#> imports
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#> module impots (adding opus)
sys.path.insert(1, '../../opus/')
import plot
import lensing
import modules.geometry as geometry

### ***The Efficacy of Morphological Arrival Ordering in Simple Mock Lenses***
---


In [128]:
#> loading data
data_dir = '../data/simple_lenses/deflect/26100115535207_nph50/'
files = glob.glob(data_dir + '*')
files = [x.replace('\\','/') for x in files]

#> print
print(f'> The files in {data_dir} are:')
x = [print(x) for x in files] # saving to object so [nones] dont print

> The files in ../data/simple_lenses/deflect/26100115535207_nph50/ are:
../data/simple_lenses/deflect/26100115535207_nph50/26100115535207_nph50_bprofiles.npy
../data/simple_lenses/deflect/26100115535207_nph50/26100115535207_nph50_df.csv
../data/simple_lenses/deflect/26100115535207_nph50/26100115535207_nph50_images.npy
../data/simple_lenses/deflect/26100115535207_nph50/26100115535207_nph50_mags.npy
../data/simple_lenses/deflect/26100115535207_nph50/26100115535207_nph50_obs.npy
../data/simple_lenses/deflect/26100115535207_nph50/26100115535207_nph50_paramRanges.npy
../data/simple_lenses/deflect/26100115535207_nph50/26100115535207_nph50_srcs.npy


In [129]:
#> loading files (by suffix, not glob order)
get    = lambda key: [x for x in files if x.endswith(key)] # matching file(s)
delays = np.load(get('_delays.npy')[0], allow_pickle=True) if get('_delays.npy') else None # time delays [days] (lpot only)
df     = pd.read_csv(get('_df.csv')[0])                    # dataframe of galaxy properties
images = np.load(get('_images.npy')[0], allow_pickle=True) # image positions [arcsec]
mags   = np.load(get('_mags.npy')[0],   allow_pickle=True) # image magnifications
obs    = np.load(get('_obs.npy')[0],    allow_pickle=True) # lensing observables
srcs   = np.load(get('_srcs.npy')[0],   allow_pickle=True) # source positions [arcsec]

#> copying
fermat_order = np.copy(images).astype(float)

In [130]:
#> getting morphological ordering & comparing
morph_order, match, perm = [], [], []
for ims in fermat_order:

    #> checking for nans (i.e., source wasn't resolved or was a triplet)
    if np.isnan(ims).any():
        morph_order.append(np.full(ims.shape, np.nan)) # filling with nan
        match.append(np.nan)                           # nan flag
        perm.append(np.full(len(ims), -1))             # error flag
        continue

    #> morphological order
    morph = lensing.arrivalOrder(images=ims, x0=0.0, y0=0.0)
    morph_order.append(morph)
    
    #> fermat index of each morph image
    idx = np.array([np.argmin(np.linalg.norm(ims - m, axis=1)) + 1 for m in morph])
    perm.append(idx)
    
    #> comparing
    match.append(float(np.all(idx == np.arange(len(ims)) + 1)))

#> formatting
morph_order = np.array(morph_order) # (numQuads, 5, 2) [arcsec]
match = np.array(match)             # 1=same order, 0=mismatch, nan=incomplete quad
perm  = np.array(perm)              # morphological sequence fermat order
valid = ~np.isnan(match)            # not nans (complete quads)

#> print
num_wrong = int(np.sum(match[valid] == 0))
print(f'> {np.sum(~valid)} quads skipped (nans), {valid.sum()} compared')
print(f'> There were {num_wrong} mismatches between morph and fermat orders!')
if num_wrong == 0: print('> Yippee!!')

> 61 quads skipped (nans), 9939 compared
> There were 0 mismatches between morph and fermat orders!
> Yippee!!


### ***Morph-Fermat Mismatches***
---

In [102]:
#> selecting mismatch images
mis = valid & (match == 0)                  # mismatch mask
mis_images = fermat_order[mis]              # (numMis, 5, 2) [arcsec]
mis_morph  = morph_order[mis]               # (numMis, 5, 2) [arcsec]
mis_perm   = perm[mis]                      # which images were swapped
mis_delays = np.array(delays[mis].tolist(), dtype=float) # (numMis, 5) [days]